# Tarea 1: prediccion de resultados del futbol uruguayo

Notebook autocontenido que lee el ZIP original, limpia los datos, construye atributos causales, guarda los datasets procesados y entrena los tres clasificadores propios (baseline de diez años, Naive Bayes con m-estimate y arbol ID3). Ejecutar todas las celdas desde la raiz del repositorio.

## 1. Configuracion reproducible

La semilla de los modelos estocasticos es 42. Las dependencias se declaran en `requirements.txt`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

ROOT = Path.cwd()
if not (ROOT / 'requirements.txt').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from baseline import TenYearWinRateClassifier  # noqa: E402
from features import (  # noqa: E402
    NUMERIC_FEATURES,
    build_causal_match_features,
    load_clean_matches,
)
from id3 import CategoricalDecisionTreeClassifier  # noqa: E402
from naive_bayes import MEstimateCategoricalNB  # noqa: E402
from preprocessing import MixedTypeDiscretizer  # noqa: E402

RAW = ROOT / 'data/raw/futbol_uruguayo.zip'
PROCESSED = ROOT / 'data/processed'
CLASSES = ['E', 'L', 'V']
LAST_5_CUTS = {
    'home_win_rate_last_5': [0.3, 0.6],
    'away_win_rate_last_5': [0.3, 0.6],
}
BIN_ORDER = ['baja', 'media', 'alta']
RANDOM_STATE = 42
pd.set_option('display.max_columns', 30)

## 2. Lectura y limpieza desde la fuente original

El ZIP contiene un unico CSV; `load_clean_matches` lo extrae y limpia en memoria. `winner` se deriva solo de `gh` y `ga`, que nunca seran atributos predictivos.

In [ ]:
matches = load_clean_matches(RAW)
display(matches.head())
print(len(matches), 'partidos')
print(
    matches['winner']
    .value_counts()
    .reindex(CLASSES)
    .rename('cantidad')
    .to_frame()
)

## 3. Atributos historicos causales

Las tasas se calculan con fechas estrictamente anteriores: todos los partidos de un dia se featurizan antes de actualizar los historiales. Los casos sin historial se imputan con el neutro documentado en `notas_decisiones.md`.

In [ ]:
featured = build_causal_match_features(matches)
train = featured[featured['year'] <= 2023].reset_index(drop=True)
test = featured[featured['year'].between(2024, 2025)].reset_index(drop=True)
display(featured[NUMERIC_FEATURES + ['winner']].head())
print(f'Train: {len(train):,} partidos ('
      f"{train['date'].min().date()} a {train['date'].max().date()})")
print(f'Test:  {len(test):,} partidos ('
      f"{test['date'].min().date()} a {test['date'].max().date()})")

## 4. Datasets procesados persistidos

- `train.csv` / `test.csv`: version numerica (tasas crudas) que consumen los modelos.
- `train_discretizado.csv` / `test_discretizado.csv`: version en tres categorias (`baja`, `media`, `alta`) solo para inspeccion visual.
- `split_report.json`: tamanos, rango de anos y distribucion de clases.

La discretizacion ajusta cuantiles **solo con train**; `win_rate_last_5` usa cortes fijos `[0.3, 0.6]`.

In [ ]:
META = ['home', 'away', 'date', 'year', 'month', 'winner']

train_numeric = train[META + NUMERIC_FEATURES].copy()
test_numeric = test[META + NUMERIC_FEATURES].copy()
for frame in (train_numeric, test_numeric):
    frame['date'] = frame['date'].dt.strftime('%Y-%m-%d')

discretizer = MixedTypeDiscretizer(
    categorical_features=[],
    numeric_features=NUMERIC_FEATURES,
    n_bins=3,
    fixed_cuts=LAST_5_CUTS,
)
discretizer.fit(train_numeric[NUMERIC_FEATURES])
train_codes = discretizer.transform(train_numeric[NUMERIC_FEATURES])
test_codes = discretizer.transform(test_numeric[NUMERIC_FEATURES])

train_disc = train_numeric[META].copy()
test_disc = test_numeric[META].copy()
for index, column in enumerate(NUMERIC_FEATURES):
    train_disc[column] = [BIN_ORDER[code - 1] for code in train_codes[:, index]]
    test_disc[column] = [BIN_ORDER[code - 1] for code in test_codes[:, index]]

PROCESSED.mkdir(parents=True, exist_ok=True)
train_numeric.to_csv(PROCESSED / 'train.csv', index=False, encoding='utf-8-sig')
test_numeric.to_csv(PROCESSED / 'test.csv', index=False, encoding='utf-8-sig')
train_disc.to_csv(
    PROCESSED / 'train_discretizado.csv', index=False, encoding='utf-8-sig'
)
test_disc.to_csv(
    PROCESSED / 'test_discretizado.csv', index=False, encoding='utf-8-sig'
)

def counts(frame: pd.DataFrame) -> dict:
    values = frame['winner'].value_counts().reindex(CLASSES)
    return {
        'rows': int(len(frame)),
        'date_range': {
            'min': frame['date'].min(),
            'max': frame['date'].max(),
        },
        'target_counts': {key: int(value) for key, value in values.items()},
    }

report = {
    'split': {'rule': 'year <= 2023 -> train; year in [2024, 2025] -> test'},
    'train': counts(train_numeric),
    'test': counts(test_numeric),
    'numeric_features': NUMERIC_FEATURES,
}
with open(PROCESSED / 'split_report.json', 'w', encoding='utf-8') as fh:
    json.dump(report, fh, indent=2, ensure_ascii=False)
    fh.write('\n')

print('Escritos:', *[path.name for path in PROCESSED.glob('*.csv')])
display(train_disc.sample(5, random_state=RANDOM_STATE))

## 5. Seleccion de hiperparametros con ventana temporal expansiva

Sobre `train` (hasta 2023) se validan valores pequeños de `m` (Naive Bayes) y `min_info_gain` (ID3) con 4 folds separados por bloques contiguos de fechas. Las discretizaciones se reajustan con el train de cada fold.

In [ ]:
def expanding_blocks(dates: pd.Series, n_blocks: int = 5):
    ordered = np.sort(dates.unique())
    blocks = np.array_split(ordered, n_blocks)
    for index in range(1, n_blocks):
        yield np.concatenate(blocks[:index]), blocks[index]


def score_discrete(estimator, fold_train: pd.DataFrame, fold_val: pd.DataFrame) -> float:
    transform = MixedTypeDiscretizer(
        categorical_features=[],
        numeric_features=NUMERIC_FEATURES,
        n_bins=3,
        fixed_cuts=LAST_5_CUTS,
    )
    transform.fit(fold_train[NUMERIC_FEATURES])
    X_train = transform.transform(fold_train[NUMERIC_FEATURES])
    X_val = transform.transform(fold_val[NUMERIC_FEATURES])
    estimator.fit(X_train, fold_train['winner'])
    predictions = estimator.predict(X_val)
    return f1_score(fold_val['winner'], predictions, average='macro')


rows = []
for fit_dates, validation_dates in expanding_blocks(train['date']):
    fit_mask = train['date'].isin(fit_dates)
    validation_mask = train['date'].isin(validation_dates)
    fold_train = train[fit_mask]
    fold_val = train[validation_mask]
    rows.append({
        'm=0.1': score_discrete(MEstimateCategoricalNB(m=0.1), fold_train, fold_val),
        'm=1.0': score_discrete(MEstimateCategoricalNB(m=1.0), fold_train, fold_val),
        'm=10': score_discrete(MEstimateCategoricalNB(m=10), fold_train, fold_val),
        'gain=0': score_discrete(
            CategoricalDecisionTreeClassifier(min_info_gain=0.0), fold_train, fold_val
        ),
        'gain=0.002': score_discrete(
            CategoricalDecisionTreeClassifier(min_info_gain=0.002), fold_train, fold_val
        ),
        'gain=0.02': score_discrete(
            CategoricalDecisionTreeClassifier(min_info_gain=0.02), fold_train, fold_val
        ),
    })
selection = pd.DataFrame(rows)
display(selection.mean().sort_values(ascending=False).round(4))

## 6. Modelos finales sobre test 2024-2025

Se reentrena cada modelo con todo `train` y se evalua sobre `test`: accuracy, macro-F1 y reporte por clase.

In [ ]:
discretizer = MixedTypeDiscretizer(
    categorical_features=[],
    numeric_features=NUMERIC_FEATURES,
    n_bins=3,
    fixed_cuts=LAST_5_CUTS,
)
discretizer.fit(train[NUMERIC_FEATURES])
X_train_disc = discretizer.transform(train[NUMERIC_FEATURES])
X_test_disc = discretizer.transform(test[NUMERIC_FEATURES])

models = {
    'baseline_10y': TenYearWinRateClassifier(),
    'naive_bayes': MEstimateCategoricalNB(m=1.0),
    'id3': CategoricalDecisionTreeClassifier(min_info_gain=0.002),
}
predictions = test[['date', 'home', 'away', 'winner']].copy()
for name, estimator in models.items():
    if name == 'baseline_10y':
        estimator.fit(train[['date', 'home', 'away']], train['winner'])
        predictions[name] = estimator.predict(test[['date', 'home', 'away']])
    else:
        estimator.fit(X_train_disc, train['winner'])
        predictions[name] = estimator.predict(X_test_disc)

summary = []
for name in models:
    expected, predicted = predictions['winner'], predictions[name]
    summary.append(
        {
            'modelo': name,
            'accuracy': round(accuracy_score(expected, predicted), 4),
            'macro_f1': round(f1_score(expected, predicted, average='macro'), 4),
        }
    )
display(pd.DataFrame(summary))
print(predictions['winner'].value_counts().reindex(CLASSES).to_string())

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(15, 4))
for axis, (name, estimator) in zip(axes, models.items()):
    expected, predicted = predictions['winner'], predictions[name]
    matrix = confusion_matrix(
        expected, predicted, labels=CLASSES
    )
    image = axis.imshow(matrix, cmap='Blues')
    axis.set_title(f'{name} (acc {accuracy_score(expected, predicted):.3f})')
    axis.set_xticks(range(3), CLASSES)
    axis.set_yticks(range(3), CLASSES)
    axis.set_xlabel('prediccion')
    axis.set_ylabel('real')
    for row in range(3):
        for column in range(3):
            axis.text(column, row, str(matrix[row, column]), ha='center', va='center')
plt.tight_layout()
plt.show()

## 7. Analisis cualitativo

Casos donde los modelos disienten y empates mal predichos para elegir ejemplos del informe.

In [ ]:
model_columns = [column for column in predictions if column in models]
predictions['aciertos'] = predictions[model_columns].eq(
    predictions['winner'], axis=0
).sum(axis=1)
display(predictions.sort_values('aciertos').head(12))
empates_reales = predictions[predictions['winner'] == 'E']
print(
    'Empates reales predichos bien por modelo:'
)
for column in model_columns:
    print(
        f'  {column}: {(empates_reales[column] == "E").mean():.1%}'
    )

## 8. Pendientes para el informe

- Registrar versiones de paquetes en el informe.
- Explicar el efecto de `m` y `min_info_gain` con la tabla de seleccion.
- Comparar todas las metricas, no solo accuracy.
- Elegir ejemplos concretos de la seccion 7.
- Discutir limitaciones: empates no predichos por el baseline, imputacion neutra y protocolo online.
- Adaptar la declaracion de uso de IA.